In [0]:
%pip install pymongo

In [0]:
dbutils.library.restartPython()

In [0]:
from common_utils.ingester import read_s3_parquet ,write_raw
from common_utils.logging import get_logger
from datetime import date
import json

# =========================================
# 0.Logger
# =========================================
logger = get_logger("func_s3_bucket.ipynb")
logger.info("session_started")


# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
logger.info("setting the configurations")
# =========================================
# 2. Read Config
# =========================================
with open(config_path,"r") as f:
    config = json.load(f)



# =========================================
# 3. CONFIG Sections
# =========================================
source_config = config['source']
target_config = config['target']
write_options_config = config['write_options']

# =========================================
# 4. Run Date
# =========================================

run_date = date.today().isoformat()
logger.info("Initialling the date")

# =========================================
# 5. Getting passwords from Databrick-Secret-Scop
# =========================================
aws_id = dbutils.secrets.get(scope="retail-platform-dev" , key =  source_config["aws_access_key_id"] )
aws_key = dbutils.secrets.get(scope="retail-platform-dev" , key =source_config["aws_secret_access_key"] )
# =========================================
# 6. READ FORM S3 Bucket
# =========================================
logger.info("Reading for S3 bucket")
df = read_s3_parquet(spark,aws_id,aws_key,source_config["path"])
logger.info("Printing the Number of rows %s", df.count())
logger.info("displing the sample data")
df.display()

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"

write_raw(df , target_path,target_config["mode"],target_config["file_format"], write_options_config["options"])
logger.info("the dataframe has been saved to the location %s", target_path)